In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [13]:
ARCHIVO = Path('../data/processed/mdt_weather_forecasting_eje.parquet')
df = pd.read_parquet(ARCHIVO)
df['fechaHora'] = pd.to_datetime(df['fechaHora'])
df['fecha'] = pd.to_datetime(df['fecha'])
df = df.set_index("fecha").sort_index()

In [3]:
df.shape

(1125813, 11)

In [4]:
df.dtypes

fechaHora             datetime64[ns]
hora                          object
estacion                      object
direccionViento              float64
humedadRelativa              float64
ozono                        float64
precipitacion                float64
presionBarometrica           float64
radiacionSolar               float64
temperaturaMedia             float64
velocidadViento              float64
dtype: object

In [14]:
STATIONS = ["Belisario", "Carapungo", "Cotocollao", "ElCamal", "LosChillos", "Tumbaco"]
VARIABLES = ["direccionViento", "humedadRelativa", "ozono", "precipitacion", "presionBarometrica", "radiacionSolar", "temperaturaMedia", "velocidadViento"]
TARGET = "temperaturaMedia"

In [15]:
# Estructura objetivo: panel (tiempo x estacion x variable)
# Se reindexa a una rejilla temporal regular COMUN a las 6 estaciones,
# para que el tensor final [T, n_estaciones, n_variables] quede alineado
def to_regular_grid_multi(df_all, stations, freq="1H", start=None, end=None):
    """df_all: DataFrame largo con columnas ['fechaHora','estacion',...vars]
       stations: lista de las 6 estaciones, en el orden que tendran los nodos del grafo
       Devuelve un dict {estacion: df_reindexado} todos sobre el MISMO indice temporal."""
    start = start or df_all["fechaHora"].min()
    end = end or df_all["fechaHora"].max()
    full_idx = pd.date_range(start, end, freq=freq)  # rejilla unica y compartida

    grids = {}
    for st in stations:
        df_st = (df_all[df_all["estacion"] == st]
                 .drop_duplicates(subset="fechaHora")
                 .set_index("fechaHora")
                 .reindex(full_idx))  # huecos -> NaN explicito, mismo largo para todas
        grids[st] = df_st
    return grids, full_idx

def panel_to_tensor(grids, stations, variables):
    """Apila los 6 dataframes en un tensor [T, n_estaciones, n_variables],
       respetando el orden de 'stations' (orden de los nodos del grafo)."""
    arrays = [grids[st][variables].to_numpy() for st in stations]
    return np.stack(arrays, axis=1)  # [T, n_nodes, n_features]

In [16]:
grids, full_idx = to_regular_grid_multi(
    df,
    stations=STATIONS,
    freq="1H"
)

In [17]:
# Validación: todas las estaciones tienen el mismo índice temporal
for st in STATIONS:
    assert len(grids[st]) == len(full_idx), f"{st} no coincide en longitud"
    assert (grids[st].index == full_idx).all(), f"{st} tiene indice desalineado"

In [18]:
print(f"Filas por estacion tras reindexar: {len(full_idx):,}")
print(f"Total esperado en tensor: {len(full_idx) * len(STATIONS):,} filas-nodo")

Filas por estacion tras reindexar: 195,024
Total esperado en tensor: 1,170,144 filas-nodo


In [19]:
# 5. Construir el tensor [T, n_nodos, n_features]
tensor = panel_to_tensor(grids, STATIONS, VARIABLES)
print("Forma del tensor:", tensor.shape)  # (T, 6, 8)

Forma del tensor: (195024, 6, 8)


In [58]:
# Construir el reporte: % NaN por estación, variable y año
records = []
years = full_idx.year

for i, st in enumerate(STATIONS):
    for j, var in enumerate(VARIABLES):
        serie = tensor[:, i, j]
        df_tmp = pd.DataFrame({"nan": np.isnan(serie), "year": years})
        pct_by_year = df_tmp.groupby("year")["nan"].mean() * 100
        for yr, pct in pct_by_year.items():
            records.append({"estacion": st, "variable": var, "anio": yr, "pct_nulos": round(pct, 2)})

df_nulos = pd.DataFrame(records)

# Pivotear: filas = (estacion, variable), columnas = año
tabla = df_nulos.pivot_table(index=["estacion", "variable"],
                             columns="anio", values="pct_nulos")

# Mostrar una estación a la vez para que sea legible
for st in STATIONS:
    print(f"\n{'='*80}")
    print(f"  {st}")
    print(f"{'='*80}")
    sub = tabla.loc[st]
    # Transponer para que los años sean filas y las variables columnas
    print(sub.T.to_string())


  Belisario
variable  direccionViento  humedadRelativa  ozono  precipitacion  presionBarometrica  radiacionSolar  temperaturaMedia  velocidadViento
anio                                                                                                                                   
2004                 4.04             4.04  15.74           4.16                4.04          100.00              4.04             4.04
2005                 2.04             2.04   6.55           2.24                2.04          100.00              2.16             2.04
2006                 0.59             0.72   3.12           0.75                1.50          100.00              0.97             0.59
2007                 8.31             0.57   2.18           0.63                0.54           33.25              0.57             0.57
2008                 2.81             1.14   3.56           0.75                1.15            1.57              1.14             2.64
2009                 1.64          

### 1. Imputación de datos vacios

In [20]:
"""
Pipeline de imputación REMMAQ - Versión final
  1. Recorte temporal a 2008-2026
  2. Detección de huecos largos (>20% anual) → máscara = 0
  3. Imputación bayesiana (IterativeImputer + BayesianRidge) para huecos cortos
  4. Validación post-imputación
"""

import numpy as np
import pandas as pd
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

STATIONS = ["Belisario", "Carapungo", "Cotocollao", "ElCamal", "LosChillos", "Tumbaco"]
VARIABLES = ["direccionViento", "humedadRelativa", "ozono", "precipitacion",
             "presionBarometrica", "radiacionSolar", "temperaturaMedia", "velocidadViento"]
UMBRAL_MASCARA = 20.0


# ============================================================
# PASO 1: RECORTE TEMPORAL
# ============================================================
def recortar_serie(tensor, full_idx, inicio="2008-01-01", fin="2026-03-31"):
    mask_tiempo = (full_idx >= inicio) & (full_idx <= fin)
    tensor_r = tensor[mask_tiempo]
    idx_r = full_idx[mask_tiempo]
    print(f"Tensor recortado: {tensor_r.shape}")
    print(f"Rango: {idx_r[0]} -> {idx_r[-1]}")
    return tensor_r, idx_r


# ============================================================
# PASO 2: MÁSCARA DE HUECOS LARGOS
# ============================================================
def construir_mascara(tensor, idx, stations, variables, umbral):
    mask = np.ones_like(tensor, dtype=np.float32)
    years = idx.year
    unique_years = np.unique(years)
    registros = []

    for i, st in enumerate(stations):
        for j, var in enumerate(variables):
            serie = tensor[:, i, j]
            for yr in unique_years:
                idx_yr = years == yr
                pct = np.isnan(serie[idx_yr]).mean() * 100
                if pct > umbral:
                    mask[idx_yr, i, j] = 0.0
                    registros.append({"estacion": st, "variable": var,
                                      "anio": yr, "pct_nulos": round(pct, 2)})

    df_log = pd.DataFrame(registros)
    if len(df_log):
        print(f"\nHuecos enmascarados ({len(df_log)} celdas estacion-variable-año):")
        print(df_log.to_string(index=False))

    pct_total = (mask == 0).mean() * 100
    print(f"\n% total enmascarado: {pct_total:.2f}%")

    # Desglose por estación
    print("\nDesglose por estación:")
    for i, st in enumerate(stations):
        pct_st = (mask[:, i, :] == 0).mean() * 100
        print(f"  {st:15s}: {pct_st:.2f}%")

    return mask, df_log


# ============================================================
# PASO 3: IMPUTACIÓN BAYESIANA
# ============================================================
def imputar_bayesiano(tensor, mask, variables, max_iter=50, random_state=42):
    tensor_imp = tensor.copy()
    stats = {}

    for j, var in enumerate(variables):
        panel = tensor[:, :, j].copy()
        mask_var = mask[:, :, j]
        nan_imputables = np.isnan(panel) & (mask_var == 1)
        n_imputables = nan_imputables.sum()

        if n_imputables == 0:
            stats[var] = {"imputados": 0, "no_imputados": 0,
                          "pct_exito": 100.0, "iteraciones": 0}
            continue

        imputer = IterativeImputer(
            estimator=BayesianRidge(),
            max_iter=max_iter,
            random_state=random_state,
            sample_posterior=False,
            tol=1e-3
        )
        panel_imputed = imputer.fit_transform(panel)

        # Solo tomar valores imputados donde mask=1
        resultado = panel.copy()
        resultado[nan_imputables] = panel_imputed[nan_imputables]
        tensor_imp[:, :, j] = resultado

        n_restantes = np.isnan(resultado[mask_var == 1]).sum()
        stats[var] = {
            "imputados": int(n_imputables - n_restantes),
            "no_imputados": int(n_restantes),
            "pct_exito": round((n_imputables - n_restantes) / max(n_imputables, 1) * 100, 2),
            "iteraciones": imputer.n_iter_
        }

    print("\nImputación bayesiana por variable:")
    print(f"  {'Variable':25s} {'Imputados':>10s} {'Restantes':>10s} {'Éxito':>8s} {'Iters':>6s}")
    print("  " + "-" * 65)
    for var, s in stats.items():
        print(f"  {var:25s} {s['imputados']:>10,} {s['no_imputados']:>10,} "
              f"{s['pct_exito']:>7.1f}% {s['iteraciones']:>5}")

    return tensor_imp, stats


# ============================================================
# PASO 4: VALIDACIÓN
# ============================================================
def validar(tensor_original, tensor_imp, mask, idx, stations, variables):
    # 4a. NaN residuales en zona imputable
    print("\n4a. NaN residuales en zona imputable (mask=1):")
    hay_residuales = False
    for i, st in enumerate(stations):
        for j, var in enumerate(variables):
            m = mask[:, i, j] == 1
            n_nan = np.isnan(tensor_imp[m, i, j]).sum()
            if n_nan > 0:
                hay_residuales = True
                print(f"  {st} / {var}: {n_nan:,} NaN residuales")
    if not hay_residuales:
        print("  Sin NaN residuales. Imputación completa.")

    # 4b. Estadísticas comparativas
    print("\n4b. Comparación estadística antes/después:")
    print(f"  {'Variable':25s} {'media_antes':>12s} {'media_desp':>12s} "
          f"{'std_antes':>10s} {'std_desp':>10s}")
    print("  " + "-" * 75)
    for j, var in enumerate(variables):
        activa = mask[:, :, j] == 1
        antes = tensor_original[:, :, j][activa]
        despues = tensor_imp[:, :, j][activa]
        a_v = antes[~np.isnan(antes)]
        d_v = despues[~np.isnan(despues)]
        print(f"  {var:25s} {a_v.mean():12.4f} {d_v.mean():12.4f} "
              f"{a_v.std():10.4f} {d_v.std():10.4f}")

    # 4c. NaN por estación en temperaturaMedia (variable objetivo)
    j_temp = variables.index("temperaturaMedia")
    print(f"\n4c. temperaturaMedia - NaN final por estación:")
    for i, st in enumerate(stations):
        m = mask[:, i, j_temp] == 1
        total = m.sum()
        n_nan = np.isnan(tensor_imp[m, i, j_temp]).sum()
        print(f"  {st:15s}: {n_nan:,} NaN de {total:,} ({n_nan/max(total,1)*100:.4f}%)")

    # 4d. Resumen global
    total_mask1 = (mask == 1).sum()
    nan_final = np.isnan(tensor_imp[mask == 1]).sum()
    print(f"\n4d. Resumen global:")
    print(f"  Celdas imputables (mask=1): {total_mask1:,}")
    print(f"  NaN finales en zona imputable: {nan_final:,} ({nan_final/total_mask1*100:.4f}%)")
    print(f"  Celdas enmascaradas (mask=0): {(mask == 0).sum():,}")


# ============================================================
# EJECUCIÓN
# ============================================================
if __name__ == "__main__":
    # Cargar tensor y eje temporal originales (con NaN, sin imputar)
    # Estos deben estar en memoria o reconstruirse desde los datos crudos:
    #   grids, full_idx = to_regular_grid_multi(df_raw, STATIONS)
    #   tensor = panel_to_tensor(grids, STATIONS, VARIABLES)



    print("=" * 60)
    print("PIPELINE DE IMPUTACIÓN REMMAQ")
    print("=" * 60)

    # Paso 1
    print("\nPASO 1: Recorte temporal (2008-2026)")
    print("-" * 40)
    tensor_r, idx_r = recortar_serie(tensor, full_idx)

    # Paso 2
    print("\nPASO 2: Máscara de huecos largos (umbral > 20%)")
    print("-" * 40)
    mask, log_huecos = construir_mascara(
        tensor_r, idx_r, STATIONS, VARIABLES, UMBRAL_MASCARA
    )

    # Paso 3
    print("\nPASO 3: Imputación bayesiana (BayesianRidge, max_iter=50)")
    print("-" * 40)
    tensor_imp, stats_imp = imputar_bayesiano(
        tensor_r, mask, VARIABLES, max_iter=50
    )

    # Paso 4
    print("\nPASO 4: Validación")
    print("-" * 40)
    validar(tensor_r, tensor_imp, mask, idx_r, STATIONS, VARIABLES)

    # Guardar artefactos
    print("\n" + "=" * 60)
    print("Guardando artefactos...")
    np.save("tensor_imputado.npy", tensor_imp)
    np.save("mascara.npy", mask)
    np.save("idx_temporal.npy", idx_r.values)
    print("  tensor_imputado.npy  - Tensor con huecos cortos imputados")
    print("  mascara.npy          - 1=dato válido/imputado, 0=hueco largo")
    print("  idx_temporal.npy     - Eje temporal (DatetimeIndex)")

PIPELINE DE IMPUTACIÓN REMMAQ

PASO 1: Recorte temporal (2008-2026)
----------------------------------------
Tensor recortado: (159937, 6, 8)
Rango: 2008-01-01 00:00:00 -> 2026-03-31 00:00:00

PASO 2: Máscara de huecos largos (umbral > 20%)
----------------------------------------

Huecos enmascarados (46 celdas estacion-variable-año):
  estacion           variable  anio  pct_nulos
 Belisario     radiacionSolar  2009      42.42
Cotocollao    humedadRelativa  2016      40.00
Cotocollao    humedadRelativa  2017     100.00
Cotocollao    humedadRelativa  2018      89.52
   ElCamal    direccionViento  2014      31.45
   ElCamal    direccionViento  2015     100.00
   ElCamal    direccionViento  2016      90.56
   ElCamal    direccionViento  2023     100.00
   ElCamal    direccionViento  2024      99.56
   ElCamal    humedadRelativa  2014      29.12
   ElCamal    humedadRelativa  2015     100.00
   ElCamal    humedadRelativa  2016      90.56
   ElCamal    humedadRelativa  2023     100.00
   E

### 2. Tratamiento de outliers

In [21]:
"""
Detección y tratamiento de outliers - Marco QA/QC REMMAQ
Alineado con US-EPA, WMO-GAW, WHO, EMEP/EEA.

Variables meteorológicas: solo rangos físicos + spike check (salto imposible).
Variables de calidad del aire (ozono): MAD(7) + percentiles p0.5/p99.5 + spike MAD(4).
Criterio integrado: invalidar solo cuando 2+ criterios se cumplan simultáneamente.
"""


STATIONS = ["Belisario", "Carapungo", "Cotocollao", "ElCamal", "LosChillos", "Tumbaco"]
VARIABLES = ["direccionViento", "humedadRelativa", "ozono", "precipitacion",
             "presionBarometrica", "radiacionSolar", "temperaturaMedia", "velocidadViento"]

# ============================================================
# RANGOS FISICOS (Marco REMMAQ, ajustados a Quito 2300-2850 m)
# ============================================================
RANGOS_FISICOS = {
    "direccionViento":    (0, 360),
    "humedadRelativa":    (0, 100),
    "ozono":              (0, 300),
    "precipitacion":      (0, 500),
    "presionBarometrica": (690, 790),   # amplio: cubre valles (Tumbaco ~771) y urbano (~726)
    "radiacionSolar":     (0, 1400),
    "temperaturaMedia":   (-5, 40),
    "velocidadViento":    (0, 40),
}

# ============================================================
# SPIKE CHECK: umbrales fijos por variable (Marco REMMAQ)
# Un "spike" es un salto imposible entre una hora y la siguiente
# ============================================================
SPIKE_THRESHOLDS = {
    "humedadRelativa":    30,    # >30% en 1 hora
    "temperaturaMedia":   10,    # >10°C en 1 hora
    "velocidadViento":    20,    # >20 m/s en 1 hora
    "radiacionSolar":     300,   # >300 W/m² en 1 lectura
    "precipitacion":      100,   # >100 mm/h en 1 hora
    "presionBarometrica": 10,    # >10 hPa en 1 hora (conservador)
    "direccionViento":    None,  # circular, no aplica spike por diferencia simple
}

# Ozono: spike via MAD temporal (|x-lag|>4*MAD AND |x-lead|>4*MAD)
OZONO_SPIKE_MAD_FACTOR = 4
OZONO_MAD_FACTOR = 7           # |x - mediana| > 7*MAD
OZONO_PERCENTILES = (0.5, 99.5)


# ============================================================
# PASO 1: RANGOS FISICOS
# ============================================================
def aplicar_rangos_fisicos(tensor, mask, variables, rangos):
    tensor_clean = tensor.copy()
    flag_rango = np.zeros_like(tensor, dtype=bool)
    reporte = []

    for j, var in enumerate(variables):
        lo, hi = rangos[var]
        datos = tensor_clean[:, :, j]
        activa = mask[:, :, j] == 1
        fuera = activa & ~np.isnan(datos) & ((datos < lo) | (datos > hi))
        n = fuera.sum()

        if n > 0:
            flag_rango[:, :, j] = fuera
            tensor_clean[:, :, j][fuera] = np.nan
            reporte.append({"variable": var, "fuera_rango": int(n),
                            "pct": round(n / activa.sum() * 100, 4)})

    print("Rangos físicos:")
    if reporte:
        print(pd.DataFrame(reporte).to_string(index=False))
    else:
        print("  Ningún valor fuera de rango.")
    return tensor_clean, flag_rango


# ============================================================
# PASO 2: SPIKE CHECK (salto imposible entre horas consecutivas)
# ============================================================
def spike_check_meteorologia(tensor, mask, variables, thresholds):
    """Spike = |x(t) - x(t-1)| > umbral AND |x(t) - x(t+1)| > umbral
    Solo para variables meteorológicas con umbral fijo."""
    flag_spike = np.zeros_like(tensor, dtype=bool)
    reporte = []

    for j, var in enumerate(variables):
        if var == "ozono":  # ozono tiene su propio spike check
            continue
        thr = thresholds.get(var)
        if thr is None:  # direccionViento
            continue

        for i in range(tensor.shape[1]):
            datos = tensor[:, i, j]
            activa = mask[:, i, j] == 1

            diff_lag = np.abs(np.diff(datos, prepend=datos[0]))
            diff_lead = np.abs(np.diff(datos, append=datos[-1]))

            # Spike: ambos saltos superan el umbral (punto aislado)
            es_spike = activa & (diff_lag > thr) & (diff_lead > thr)
            # No marcar el primer y último punto
            es_spike[0] = False
            es_spike[-1] = False

            n = es_spike.sum()
            if n > 0:
                flag_spike[:, i, j] = es_spike
                reporte.append({"estacion": STATIONS[i], "variable": var,
                                "spikes": int(n), "umbral": thr})

    print("\nSpike check meteorológico:")
    if reporte:
        print(pd.DataFrame(reporte).to_string(index=False))
    else:
        print("  Ningún spike detectado.")
    return flag_spike


# ============================================================
# PASO 3: QA/QC OZONO (MAD + percentiles + spike temporal)
# ============================================================
def qaqc_ozono(tensor, mask, variables):
    """Tres criterios para ozono, se invalida con 2+ simultáneos."""
    j = variables.index("ozono")

    flag_mad = np.zeros(tensor.shape[:2], dtype=bool)
    flag_pct = np.zeros(tensor.shape[:2], dtype=bool)
    flag_spike = np.zeros(tensor.shape[:2], dtype=bool)
    reporte = []

    for i in range(tensor.shape[1]):
        datos = tensor[:, i, j]
        activa = mask[:, i, j] == 1
        validos = activa & ~np.isnan(datos)

        if validos.sum() == 0:
            continue

        vals = datos[validos]
        med = np.nanmedian(vals)
        mad = np.nanmedian(np.abs(vals - med))

        # Criterio 1: MAD con factor 7
        if mad > 0:
            zscore_mod = np.abs(datos - med) / mad
            flag_mad[:, i] = activa & (zscore_mod > OZONO_MAD_FACTOR)

        # Criterio 2: percentiles p0.5 y p99.5
        p_lo = np.percentile(vals, OZONO_PERCENTILES[0])
        p_hi = np.percentile(vals, OZONO_PERCENTILES[1])
        flag_pct[:, i] = activa & ~np.isnan(datos) & ((datos < p_lo) | (datos > p_hi))

        # Criterio 3: spike temporal |x-lag|>4*MAD AND |x-lead|>4*MAD
        if mad > 0:
            thr_spike = OZONO_SPIKE_MAD_FACTOR * mad
            diff_lag = np.abs(np.diff(datos, prepend=datos[0]))
            diff_lead = np.abs(np.diff(datos, append=datos[-1]))
            spike = activa & (diff_lag > thr_spike) & (diff_lead > thr_spike)
            spike[0] = False
            spike[-1] = False
            flag_spike[:, i] = spike

        # Criterio integrado: 2+ de 3 criterios
        n_criterios = flag_mad[:, i].astype(int) + flag_pct[:, i].astype(int) + flag_spike[:, i].astype(int)
        outliers_integrado = n_criterios >= 2

        n_mad = flag_mad[:, i].sum()
        n_pct = flag_pct[:, i].sum()
        n_spk = flag_spike[:, i].sum()
        n_final = outliers_integrado.sum()

        reporte.append({
            "estacion": STATIONS[i],
            "flag_MAD7": int(n_mad),
            "flag_p05_p995": int(n_pct),
            "flag_spike4MAD": int(n_spk),
            "outliers_final_2+": int(n_final),
            "pct": round(n_final / validos.sum() * 100, 4)
        })

        # Sobrescribir flags individuales con el criterio integrado
        flag_mad[:, i] = outliers_integrado

    print("\nQA/QC Ozono (criterio integrado: 2+ de 3 flags):")
    print(pd.DataFrame(reporte).to_string(index=False))

    # Devolver como flag en la posición de ozono del tensor
    flag_ozono = np.zeros_like(tensor, dtype=bool)
    flag_ozono[:, :, j] = flag_mad  # ya contiene el criterio integrado
    return flag_ozono


# ============================================================
# PASO 4: COMBINAR FLAGS Y REEMPLAZAR
# ============================================================
def combinar_y_reemplazar(tensor, mask, flag_rango, flag_spike, flag_ozono):
    """Combina todos los flags, convierte outliers a NaN,
    y los rellena con interpolación temporal."""

    flag_total = flag_rango | flag_spike | flag_ozono
    n_total = flag_total.sum()

    tensor_clean = tensor.copy()
    tensor_clean[flag_total] = np.nan

    # Interpolación temporal por estación-variable
    n_reparados = 0
    for i in range(tensor.shape[1]):
        for j in range(tensor.shape[2]):
            serie = pd.Series(tensor_clean[:, i, j])
            nan_antes = serie.isna().sum()

            serie = serie.interpolate(method="linear", limit=6, limit_direction="both")

            if serie.isna().any():
                mediana_movil = serie.rolling(48, min_periods=6, center=True).median()
                serie = serie.fillna(mediana_movil)

            nan_despues = serie.isna().sum()
            n_reparados += (nan_antes - nan_despues)
            tensor_clean[:, i, j] = serie.values

    # Restaurar NaN donde mask=0
    tensor_clean[mask == 0] = np.nan

    print(f"\nOutliers totales marcados: {n_total:,}")
    print(f"  - Por rango físico: {flag_rango.sum():,}")
    print(f"  - Por spike meteorológico: {flag_spike.sum():,}")
    print(f"  - Por QA/QC ozono (2+ criterios): {flag_ozono.sum():,}")
    print(f"Reparados por interpolación: {n_reparados:,}")

    nan_residual = np.isnan(tensor_clean[mask == 1]).sum()
    print(f"NaN residuales en zona imputable: {nan_residual:,}")

    return tensor_clean, flag_total


# ============================================================
# PASO 5: VALIDACION
# ============================================================
def validar(tensor_original, tensor_clean, mask, variables):
    print("\nComparación antes/después (zona imputable):")
    print("=" * 90)
    print(f"{'Variable':25s} {'media_antes':>12s} {'media_desp':>12s} {'std_antes':>10s} "
          f"{'std_desp':>10s} {'max_antes':>10s} {'max_desp':>10s}")
    print("-" * 90)

    for j, var in enumerate(variables):
        activa = mask[:, :, j] == 1
        antes = tensor_original[:, :, j][activa]
        despues = tensor_clean[:, :, j][activa]
        antes_v = antes[~np.isnan(antes)]
        despues_v = despues[~np.isnan(despues)]

        print(f"{var:25s} {antes_v.mean():12.4f} {despues_v.mean():12.4f} "
              f"{antes_v.std():10.4f} {despues_v.std():10.4f} "
              f"{antes_v.max():10.4f} {despues_v.max():10.4f}")


# ============================================================
# EJECUCION
# ============================================================
if __name__ == "__main__":
    tensor_imp = np.load("tensor_imputado.npy")
    mask = np.load("mascara.npy")
    idx_r = pd.DatetimeIndex(np.load("idx_temporal.npy"))

    print("PASO 1: Rangos físicos (Marco REMMAQ)")
    print("=" * 50)
    tensor_rango, flag_rango = aplicar_rangos_fisicos(
        tensor_imp, mask, VARIABLES, RANGOS_FISICOS
    )

    print("\nPASO 2: Spike check meteorológico")
    print("=" * 50)
    flag_spike = spike_check_meteorologia(
        tensor_rango, mask, VARIABLES, SPIKE_THRESHOLDS
    )

    print("\nPASO 3: QA/QC Ozono (MAD + percentiles + spike)")
    print("=" * 50)
    flag_ozono = qaqc_ozono(tensor_rango, mask, VARIABLES)

    print("\nPASO 4: Combinación y reemplazo")
    print("=" * 50)
    tensor_clean, flag_total = combinar_y_reemplazar(
        tensor_rango, mask, flag_rango, flag_spike, flag_ozono
    )

    print("\nPASO 5: Validación")
    print("=" * 50)
    validar(tensor_imp, tensor_clean, mask, VARIABLES)

    np.save("tensor_clean.npy", tensor_clean)
    print("\nGuardado: tensor_clean.npy")

PASO 1: Rangos físicos (Marco REMMAQ)
Rangos físicos:
       variable  fuera_rango    pct
humedadRelativa            1 0.0001
          ozono          435 0.0480
 radiacionSolar          585 0.0645

PASO 2: Spike check meteorológico

Spike check meteorológico:
  estacion           variable  spikes  umbral
 Carapungo presionBarometrica       1      10
 Belisario     radiacionSolar    1323     300
 Carapungo     radiacionSolar    1571     300
Cotocollao     radiacionSolar    1952     300
   ElCamal     radiacionSolar    1090     300
LosChillos     radiacionSolar    1842     300
   Tumbaco     radiacionSolar    1462     300
Cotocollao   temperaturaMedia       1      10

PASO 3: QA/QC Ozono (MAD + percentiles + spike)

QA/QC Ozono (criterio integrado: 2+ de 3 flags):
  estacion  flag_MAD7  flag_p05_p995  flag_spike4MAD  outliers_final_2+    pct
 Belisario        100           1582               3                100 0.0625
 Carapungo         35           1585               0                

### 3. Feature engineering

In [22]:
"""
Feature Engineering Temporal para STGNN REMMAQ.
Agrega codificación cíclica (sin/cos) al tensor limpio.

Entrada:  tensor_clean.npy [T, 6, 8]   (8 variables climáticas)
Salida:   tensor_features.npy [T, 6, 14] (8 climáticas + 6 cíclicas)

Las features temporales son idénticas para las 6 estaciones en cada
instante (misma hora, mismo día), pero se replican en la dimensión
de nodos para mantener la forma [T, nodos, features] que el STGNN espera.
"""

STATIONS = ["Belisario", "Carapungo", "Cotocollao", "ElCamal", "LosChillos", "Tumbaco"]

VARIABLES_ORIG = [
    "direccionViento", "humedadRelativa", "ozono", "precipitacion",
    "presionBarometrica", "radiacionSolar", "temperaturaMedia", "velocidadViento"
]

VARIABLES_CICLICAS = [
    "hour_sin", "hour_cos",
    "dow_sin", "dow_cos",
    "doy_sin", "doy_cos"
]

VARIABLES_FULL = VARIABLES_ORIG + VARIABLES_CICLICAS


def crear_features_ciclicas(idx):
    """Genera 6 features cíclicas a partir del DatetimeIndex.
    Retorna array [T, 6]."""
    hour = idx.hour
    dow = idx.dayofweek
    doy = idx.dayofyear

    features = np.column_stack([
        np.sin(2 * np.pi * hour / 24),
        np.cos(2 * np.pi * hour / 24),
        np.sin(2 * np.pi * dow / 7),
        np.cos(2 * np.pi * dow / 7),
        np.sin(2 * np.pi * doy / 365),
        np.cos(2 * np.pi * doy / 365),
    ])
    return features  # [T, 6]


def expandir_a_nodos(features_temporales, n_nodos):
    """Replica las features temporales [T, n_feat] para cada nodo.
    Retorna [T, n_nodos, n_feat]."""
    return np.repeat(features_temporales[:, np.newaxis, :], n_nodos, axis=1)


def expandir_mascara(mask, n_feat_nuevas):
    """La máscara original es [T, 6, 8]. Las features cíclicas nunca
    son NaN (se calculan desde el timestamp), así que se extiende
    con 1s en las nuevas columnas."""
    extension = np.ones((*mask.shape[:2], n_feat_nuevas), dtype=mask.dtype)
    return np.concatenate([mask, extension], axis=2)


if __name__ == "__main__":
    tensor_clean = np.load("tensor_clean.npy")
    mask = np.load("mascara.npy")
    idx_r = pd.DatetimeIndex(np.load("idx_temporal.npy"))

    print(f"Tensor entrada: {tensor_clean.shape}")  # [T, 6, 8]
    print(f"Rango: {idx_r[0]} -> {idx_r[-1]}")

    # Crear features cíclicas
    feat_ciclicas = crear_features_ciclicas(idx_r)  # [T, 6]
    print(f"\nFeatures cíclicas generadas: {feat_ciclicas.shape}")

    # Verificación de rangos (sin/cos siempre en [-1, 1])
    print(f"  Rango min: {feat_ciclicas.min():.4f}")
    print(f"  Rango max: {feat_ciclicas.max():.4f}")

    # Expandir a 6 nodos
    feat_expandidas = expandir_a_nodos(feat_ciclicas, len(STATIONS))  # [T, 6, 6]
    print(f"  Expandidas a nodos: {feat_expandidas.shape}")

    # Concatenar al tensor
    tensor_full = np.concatenate([tensor_clean, feat_expandidas], axis=2)  # [T, 6, 14]
    print(f"\nTensor final: {tensor_full.shape}")

    # Expandir máscara
    mask_full = expandir_mascara(mask, len(VARIABLES_CICLICAS))
    print(f"Máscara final: {mask_full.shape}")

    # Resumen de variables
    print(f"\nVariables ({len(VARIABLES_FULL)}):")
    for i, var in enumerate(VARIABLES_FULL):
        origen = "climática" if i < len(VARIABLES_ORIG) else "cíclica"
        print(f"  [{i:2d}] {var:25s} ({origen})")

    # Índice de la variable objetivo
    idx_target = VARIABLES_FULL.index("temperaturaMedia")
    print(f"\nÍndice de temperaturaMedia (target): {idx_target}")

    # Guardar
    np.save("tensor_features.npy", tensor_full)
    np.save("mascara_features.npy", mask_full)
    print("\nGuardados: tensor_features.npy, mascara_features.npy")

Tensor entrada: (159937, 6, 8)
Rango: 2008-01-01 00:00:00 -> 2026-03-31 00:00:00

Features cíclicas generadas: (159937, 6)
  Rango min: -1.0000
  Rango max: 1.0000
  Expandidas a nodos: (159937, 6, 6)

Tensor final: (159937, 6, 14)
Máscara final: (159937, 6, 14)

Variables (14):
  [ 0] direccionViento           (climática)
  [ 1] humedadRelativa           (climática)
  [ 2] ozono                     (climática)
  [ 3] precipitacion             (climática)
  [ 4] presionBarometrica        (climática)
  [ 5] radiacionSolar            (climática)
  [ 6] temperaturaMedia          (climática)
  [ 7] velocidadViento           (climática)
  [ 8] hour_sin                  (cíclica)
  [ 9] hour_cos                  (cíclica)
  [10] dow_sin                   (cíclica)
  [11] dow_cos                   (cíclica)
  [12] doy_sin                   (cíclica)
  [13] doy_cos                   (cíclica)

Índice de temperaturaMedia (target): 6

Guardados: tensor_features.npy, mascara_features.npy


### 3. Normalización y partición temporal

In [25]:
"""
Preparación final del dataset para STGNN REMMAQ.
  1. Partición temporal: train / val / test (cronológica)
  2. Normalización: StandardScaler ajustado solo en train
  3. Matriz de adyacencia: Haversine + kernel gaussiano umbralizado
  4. Ventaneo: sliding windows + DataLoaders para GPU
"""

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

STATIONS = ["Belisario", "Carapungo", "Cotocollao", "ElCamal", "LosChillos", "Tumbaco"]

VARIABLES_FULL = [
    "direccionViento", "humedadRelativa", "ozono", "precipitacion",
    "presionBarometrica", "radiacionSolar", "temperaturaMedia", "velocidadViento",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos", "doy_sin", "doy_cos"
]

TARGET_IDX = VARIABLES_FULL.index("temperaturaMedia")  # 6
N_NODES = len(STATIONS)
N_FEATURES = len(VARIABLES_FULL)

# Coordenadas exactas de las estaciones (fuente: Secretaría de Ambiente DMQ)
COORDS = {
    "Belisario":  (-0.185170, -78.495677),
    "Carapungo":  (-0.095393, -78.449755),
    "Cotocollao":  (-0.107716, -78.497268),
    "ElCamal":    (-0.249970, -78.510058),
    "LosChillos": (-0.297100, -78.455270),
    "Tumbaco":    (-0.215015, -78.403442),
}


# ============================================================
# PASO 1: PARTICIÓN TEMPORAL
# ============================================================
def partir_temporal(tensor, mask, idx, train_end="2021-12-31 23:00:00",
                    val_end="2023-12-31 23:00:00"):
    """División cronológica estricta. No hay shuffle ni aleatoriedad.
    Train: 2008-2021 (14 años)
    Val:   2022-2023 (2 años)
    Test:  2024-2026 (2.25 años)"""

    m_train = idx <= train_end
    m_val = (idx > train_end) & (idx <= val_end)
    m_test = idx > val_end

    splits = {
        "train": (tensor[m_train], mask[m_train], idx[m_train]),
        "val":   (tensor[m_val],   mask[m_val],   idx[m_val]),
        "test":  (tensor[m_test],  mask[m_test],  idx[m_test]),
    }

    print("Partición temporal:")
    for name, (t, m, i) in splits.items():
        pct_mask0 = (m[:, :, :8] == 0).mean() * 100  # solo variables climáticas
        print(f"  {name:5s}: {i[0].strftime('%Y-%m-%d')} -> {i[-1].strftime('%Y-%m-%d')} "
              f"| {t.shape[0]:,} pasos | {pct_mask0:.1f}% enmascarado")

    return splits


# ============================================================
# PASO 2: NORMALIZACIÓN
# ============================================================
def crear_escaladores(train_tensor, train_mask, n_features):
    """Ajusta un StandardScaler por feature, solo con datos de train
    donde mask=1. Las features cíclicas (idx 8-13) ya están en [-1,1],
    se normalizan igual para consistencia."""

    scalers = []
    for f in range(n_features):
        sc = StandardScaler()
        # Extraer todos los valores válidos de esta feature en train
        datos = train_tensor[:, :, f]
        mascara = train_mask[:, :, f] == 1
        valores_validos = datos[mascara].reshape(-1, 1)
        valores_validos = valores_validos[~np.isnan(valores_validos).flatten()]
        sc.fit(valores_validos.reshape(-1, 1))
        scalers.append(sc)
    return scalers


def aplicar_escaladores(tensor, scalers):
    """Aplica los scalers (ajustados en train) a cualquier split."""
    tensor_scaled = tensor.copy()
    for f, sc in enumerate(scalers):
        datos = tensor_scaled[:, :, f]
        shape_orig = datos.shape
        datos_flat = datos.reshape(-1, 1)
        datos_scaled = sc.transform(datos_flat).reshape(shape_orig)
        tensor_scaled[:, :, f] = datos_scaled
    return tensor_scaled


def invertir_escalado_target(valores, scalers, target_idx):
    """Desnormaliza predicciones del target para calcular métricas
    en unidades originales (°C)."""
    sc = scalers[target_idx]
    return sc.inverse_transform(valores.reshape(-1, 1)).flatten()


# ============================================================
# PASO 3: MATRIZ DE ADYACENCIA
# ============================================================
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2)**2 + np.cos(p1) * np.cos(p2) * np.sin(dlam / 2)**2
    return 2 * R * np.arcsin(np.sqrt(a))


def construir_adyacencia(stations, coords, threshold=0.5):
    """Kernel gaussiano umbralizado sobre distancia de Haversine.
    A_ij = exp(-d_ij² / σ²) si >= threshold, else 0.
    σ = std de todas las distancias entre pares."""

    n = len(stations)
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            lat1, lon1 = coords[stations[i]]
            lat2, lon2 = coords[stations[j]]
            D[i, j] = haversine_km(lat1, lon1, lat2, lon2)

    sigma = D[D > 0].mean()
    A = np.exp(-(D**2) / (sigma**2))
    A[A < threshold] = 0.0
    np.fill_diagonal(A, 0.0)  # sin auto-loops

    print("\nMatriz de distancias (km):")
    df_dist = pd.DataFrame(D, index=stations, columns=stations).round(2)
    print(df_dist)

    print(f"\nσ (std distancias): {sigma:.2f} km")
    print(f"Threshold: {threshold}")

    print("\nMatriz de adyacencia (pesos):")
    df_adj = pd.DataFrame(A, index=stations, columns=stations).round(4)
    print(df_adj)

    # Conexiones activas
    n_edges = (A > 0).sum()
    print(f"\nAristas activas: {n_edges} de {n * (n - 1)} posibles")

    return A, D


def adyacencia_a_edges(A):
    """Convierte matriz de adyacencia a edge_index y edge_weight
    para PyTorch Geometric."""
    src, dst = np.nonzero(A)
    edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long)
    edge_weight = torch.tensor(A[src, dst], dtype=torch.float)
    return edge_index, edge_weight


# ============================================================
# PASO 4: VENTANEO Y DATALOADERS
# ============================================================
class STGNNDataset(Dataset):
    """Genera ventanas deslizantes para STGNN.
    X: [seq_len, n_nodes, n_features]
    y: [horizon, n_nodes]  (solo temperatura)
    m: [horizon, n_nodes]  (máscara del target para el loss)"""

    def __init__(self, data, mask, seq_len, horizon, target_idx):
        self.data = data
        self.mask = mask
        self.seq_len = seq_len
        self.horizon = horizon
        self.target_idx = target_idx
        self.n_samples = data.shape[0] - seq_len - horizon + 1

    def __len__(self):
        return self.n_samples

    def __getitem__(self, i):
        X = self.data[i:i + self.seq_len]                                    # [seq_len, nodes, feat]
        y = self.data[i + self.seq_len:i + self.seq_len + self.horizon,
                      :, self.target_idx]                                    # [horizon, nodes]
        m = self.mask[i + self.seq_len:i + self.seq_len + self.horizon,
                      :, self.target_idx]                                    # [horizon, nodes]

        return (torch.tensor(X, dtype=torch.float32),
                torch.tensor(y, dtype=torch.float32),
                torch.tensor(m, dtype=torch.float32))


def crear_dataloaders(splits_scaled, masks, seq_len, horizon, batch_size,
                      target_idx, num_workers=2):
    """Crea DataLoaders para train, val y test."""

    loaders = {}
    for name in ["train", "val", "test"]:
        data, mask = splits_scaled[name], masks[name]
        ds = STGNNDataset(data, mask, seq_len, horizon, target_idx)
        shuffle = (name == "train")
        loaders[name] = DataLoader(
            ds, batch_size=batch_size, shuffle=shuffle,
            num_workers=num_workers, pin_memory=True, drop_last=False
        )
        print(f"  {name:5s}: {len(ds):,} ventanas | {len(loaders[name]):,} batches")

    return loaders


# ============================================================
# EJECUCIÓN
# ============================================================
if __name__ == "__main__":
    tensor = np.load("tensor_features.npy")
    mask = np.load("mascara_features.npy")
    idx = pd.DatetimeIndex(np.load("idx_temporal.npy"))

    print("=" * 60)
    print("PREPARACIÓN FINAL DEL DATASET PARA STGNN")
    print("=" * 60)
    print(f"Tensor entrada: {tensor.shape}")
    print(f"Target: {VARIABLES_FULL[TARGET_IDX]} (índice {TARGET_IDX})")

    # --- Paso 1: Partición temporal ---
    print("\nPASO 1: Partición temporal")
    print("-" * 40)
    splits = partir_temporal(tensor, mask, idx)

    # --- Paso 2: Normalización ---
    print("\nPASO 2: Normalización (fit solo en train)")
    print("-" * 40)
    train_t, train_m, _ = splits["train"]
    scalers = crear_escaladores(train_t, train_m, N_FEATURES)

    splits_scaled = {}
    masks_split = {}
    for name, (t, m, i) in splits.items():
        t_scaled = aplicar_escaladores(t, scalers)
        # Restaurar NaN donde mask=0 (la normalización puede haber puesto valores)
        t_scaled[m == 0] = 0.0  # valor neutro para el modelo en zonas enmascaradas
        splits_scaled[name] = t_scaled
        masks_split[name] = m

    # Verificación: media y std en train deben ser ~0 y ~1
    train_scaled = splits_scaled["train"]
    train_mask = masks_split["train"]
    print("\nVerificación (train, zona imputable):")
    print(f"  {'Variable':25s} {'media':>8s} {'std':>8s}")
    for f in range(min(N_FEATURES, 8)):  # solo climáticas
        vals = train_scaled[:, :, f][train_mask[:, :, f] == 1]
        print(f"  {VARIABLES_FULL[f]:25s} {vals.mean():8.4f} {vals.std():8.4f}")

    # --- Paso 3: Matriz de adyacencia ---
    print("\nPASO 3: Matriz de adyacencia (Haversine + Gaussiana)")
    print("-" * 40)
    A, D = construir_adyacencia(STATIONS, COORDS, threshold=0.0)
    edge_index, edge_weight = adyacencia_a_edges(A)
    print(f"\nedge_index shape: {edge_index.shape}")
    print(f"edge_weight shape: {edge_weight.shape}")

    # --- Paso 4: Ventaneo y DataLoaders ---
    # Configuración de ventanas de predicción
    SEQ_LEN = 24       # 24 horas de historia
    HORIZON = 1         # predecir 1 hora adelante (ajustar para 6h, 24h)
    BATCH_SIZE = 32

    print(f"\nPASO 4: Ventaneo (seq_len={SEQ_LEN}, horizon={HORIZON}, batch={BATCH_SIZE})")
    print("-" * 40)
    loaders = crear_dataloaders(
        splits_scaled, masks_split,
        seq_len=SEQ_LEN, horizon=HORIZON,
        batch_size=BATCH_SIZE, target_idx=TARGET_IDX
    )

    # Verificar forma de un batch
    X_sample, y_sample, m_sample = next(iter(loaders["train"]))
    print(f"\nForma de un batch:")
    print(f"  X: {X_sample.shape}")   # [batch, seq_len, nodes, features]
    print(f"  y: {y_sample.shape}")   # [batch, horizon, nodes]
    print(f"  mask: {m_sample.shape}") # [batch, horizon, nodes]

    # --- Guardar artefactos ---
    print("\n" + "=" * 60)
    print("Guardando artefactos...")
    torch.save(edge_index, "edge_index.pt")
    torch.save(edge_weight, "edge_weight.pt")

    # Guardar splits normalizados y máscaras para reproducibilidad
    for name in ["train", "val", "test"]:
        np.save(f"tensor_{name}.npy", splits_scaled[name])
        np.save(f"mask_{name}.npy", masks_split[name])

    # Guardar scalers para desnormalizar predicciones
    import pickle
    with open("scalers.pkl", "wb") as f:
        pickle.dump(scalers, f)

    print("  edge_index.pt, edge_weight.pt")
    print("  tensor_train.npy, tensor_val.npy, tensor_test.npy")
    print("  mask_train.npy, mask_val.npy, mask_test.npy")
    print("  scalers.pkl")
    print(f"\nConfig para el modelo:")
    print(f"  in_channels = {N_FEATURES}")
    print(f"  n_nodes = {N_NODES}")
    print(f"  seq_len = {SEQ_LEN}")
    print(f"  horizon = {HORIZON}")
    print(f"  target_idx = {TARGET_IDX}")

PREPARACIÓN FINAL DEL DATASET PARA STGNN
Tensor entrada: (159937, 6, 14)
Target: temperaturaMedia (índice 6)

PASO 1: Partición temporal
----------------------------------------
Partición temporal:
  train: 2008-01-01 -> 2021-12-31 | 122,736 pasos | 4.3% enmascarado
  val  : 2022-01-01 -> 2023-12-31 | 17,520 pasos | 8.3% enmascarado
  test : 2024-01-01 -> 2026-03-31 | 19,681 pasos | 8.4% enmascarado

PASO 2: Normalización (fit solo en train)
----------------------------------------

Verificación (train, zona imputable):
  Variable                     media      std
  direccionViento            -0.0000   1.0000
  humedadRelativa            -0.0000   1.0000
  ozono                       0.0000   1.0000
  precipitacion               0.0000   1.0000
  presionBarometrica          0.0000   1.0000
  radiacionSolar              0.0000   1.0000
  temperaturaMedia           -0.0000   1.0000
  velocidadViento            -0.0000   1.0000

PASO 3: Matriz de adyacencia (Haversine + Gaussiana)
------

### 4. Entrenamiento y definición del modelo. A3-GCN

In [28]:
"""
Entrenamiento A3T-GCN - REMMAQ (versión optimizada)
Batching a nivel de grafo: elimina el loop per-sample de Python.
"""

import numpy as np
import pandas as pd
import pickle
import time
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from torch_geometric_temporal.nn.recurrent import A3TGCN

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CONFIG = {
    "n_nodes": 6,
    "in_channels": 14,
    "hidden": 64,
    "seq_len": 24,
    "horizon": 1,
    "target_idx": 6,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "epochs": 100,
    "batch_size": 32,
    "patience": 15,
    "seed": 42,
}


def set_seed(seed):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# DATASET
# ============================================================
class STGNNDataset(Dataset):
    def __init__(self, data, mask, seq_len, horizon, target_idx):
        self.data = data
        self.mask = mask
        self.seq_len = seq_len
        self.horizon = horizon
        self.target_idx = target_idx
        self.n_samples = data.shape[0] - seq_len - horizon + 1

    def __len__(self):
        return self.n_samples

    def __getitem__(self, i):
        X = self.data[i:i + self.seq_len]
        y = self.data[i + self.seq_len:i + self.seq_len + self.horizon,
                      :, self.target_idx]
        m = self.mask[i + self.seq_len:i + self.seq_len + self.horizon,
                      :, self.target_idx]
        return (torch.tensor(X, dtype=torch.float32),
                torch.tensor(y, dtype=torch.float32),
                torch.tensor(m, dtype=torch.float32))


# ============================================================
# BATCHING DE GRAFO
# ============================================================
def batch_edge_index(edge_index, batch_size, n_nodes):
    """Replica edge_index para cada muestra del batch, desplazando
    los índices de nodo. Resultado: un super-grafo de batch_size
    copias desconectadas del grafo original."""
    edges = []
    for i in range(batch_size):
        edges.append(edge_index + i * n_nodes)
    return torch.cat(edges, dim=1)


def batch_edge_weight(edge_weight, batch_size):
    """Replica edge_weight para cada copia del grafo."""
    return edge_weight.repeat(batch_size)


# ============================================================
# MODELO
# ============================================================
class A3TGCNForecaster(nn.Module):
    def __init__(self, in_channels, hidden, horizon, periods):
        super().__init__()
        self.tgnn = A3TGCN(
            in_channels=in_channels,
            out_channels=hidden,
            periods=periods
        )
        self.head = nn.Linear(hidden, horizon)

    def forward(self, x, edge_index, edge_weight):
        # x: [total_nodes, in_channels, periods]
        h = self.tgnn(x, edge_index, edge_weight)  # [total_nodes, hidden]
        return self.head(h)                          # [total_nodes, horizon]


# ============================================================
# LOSS ENMASCARADO
# ============================================================
def masked_mae_loss(pred, target, mask):
    diff = torch.abs(pred - target) * mask
    n_valid = mask.sum()
    if n_valid == 0:
        return torch.tensor(0.0, device=pred.device, requires_grad=True)
    return diff.sum() / n_valid


# ============================================================
# ENTRENAMIENTO
# ============================================================
def train_epoch(model, loader, optimizer, scaler, edge_index_base,
                edge_weight_base, n_nodes):
    model.train()
    total_loss = 0.0
    n_batches = 0
    optimizer.zero_grad()

    for X, y, m in loader:
        bs = X.size(0)  # batch size real (último batch puede ser menor)
        X = X.to(DEVICE)
        y = y.to(DEVICE)
        m = m.to(DEVICE)

        # Construir super-grafo para este batch
        ei = batch_edge_index(edge_index_base, bs, n_nodes).to(DEVICE)
        ew = batch_edge_weight(edge_weight_base, bs).to(DEVICE)

        with autocast():
            # X: [batch, seq_len, nodes, feat] -> [batch*nodes, feat, seq_len]
            x_in = X.permute(0, 2, 3, 1)             # [batch, nodes, feat, seq_len]
            x_in = x_in.reshape(bs * n_nodes, -1, X.size(1))  # [batch*nodes, feat, seq_len]

            out = model(x_in, ei, ew)                  # [batch*nodes, horizon]
            out = out.reshape(bs, n_nodes, -1)         # [batch, nodes, horizon]
            out = out.permute(0, 2, 1)                 # [batch, horizon, nodes]

            loss = masked_mae_loss(out, y, m)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / n_batches


# ============================================================
# EVALUACIÓN
# ============================================================
@torch.no_grad()
def evaluate(model, loader, edge_index_base, edge_weight_base, n_nodes,
             scalers, target_idx):
    model.eval()
    all_pred, all_target, all_mask = [], [], []

    for X, y, m in loader:
        bs = X.size(0)
        X = X.to(DEVICE)

        ei = batch_edge_index(edge_index_base, bs, n_nodes).to(DEVICE)
        ew = batch_edge_weight(edge_weight_base, bs).to(DEVICE)

        with autocast():
            x_in = X.permute(0, 2, 3, 1)
            x_in = x_in.reshape(bs * n_nodes, -1, X.size(1))
            out = model(x_in, ei, ew)
            out = out.reshape(bs, n_nodes, -1).permute(0, 2, 1)

        all_pred.append(out.cpu())
        all_target.append(y)
        all_mask.append(m)

    pred = torch.cat(all_pred)
    target = torch.cat(all_target)
    mask = torch.cat(all_mask)

    loss_norm = masked_mae_loss(pred, target, mask).item()

    sc = scalers[target_idx]
    pred_np = pred.numpy().flatten()
    target_np = target.numpy().flatten()
    mask_np = mask.numpy().flatten().astype(bool)

    pred_real = sc.inverse_transform(pred_np.reshape(-1, 1)).flatten()
    target_real = sc.inverse_transform(target_np.reshape(-1, 1)).flatten()
    pred_real = pred_real[mask_np]
    target_real = target_real[mask_np]

    mae = np.mean(np.abs(target_real - pred_real))
    rmse = np.sqrt(np.mean((target_real - pred_real) ** 2))
    ss_res = np.sum((target_real - pred_real) ** 2)
    ss_tot = np.sum((target_real - target_real.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot

    return {"loss_norm": loss_norm, "MAE_C": mae, "RMSE_C": rmse, "R2": r2}


# ============================================================
# EJECUCIÓN
# ============================================================
if __name__ == "__main__":
    set_seed(CONFIG["seed"])

    print("=" * 60)
    print("ENTRENAMIENTO A3T-GCN - REMMAQ (OPTIMIZADO)")
    print("=" * 60)
    print(f"Device: {DEVICE}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

    # Cargar artefactos
    print("\nCargando artefactos...")
    edge_index = torch.load("edge_index.pt")
    edge_weight = torch.load("edge_weight.pt")

    with open("scalers.pkl", "rb") as f:
        scalers = pickle.load(f)

    loaders = {}
    for split in ["train", "val", "test"]:
        data = np.load(f"tensor_{split}.npy")
        mask = np.load(f"mask_{split}.npy")
        ds = STGNNDataset(data, mask, CONFIG["seq_len"], CONFIG["horizon"],
                          CONFIG["target_idx"])
        shuffle = (split == "train")
        loaders[split] = DataLoader(
            ds, batch_size=CONFIG["batch_size"], shuffle=shuffle,
            num_workers=2, pin_memory=True, drop_last=False
        )
        print(f"  {split:5s}: {len(ds):,} ventanas")

    # Modelo
    model = A3TGCNForecaster(
        in_channels=CONFIG["in_channels"],
        hidden=CONFIG["hidden"],
        horizon=CONFIG["horizon"],
        periods=CONFIG["seq_len"]
    ).to(DEVICE)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nModelo: A3T-GCN (batched)")
    print(f"  Parámetros entrenables: {n_params:,}")
    print(f"  Hidden: {CONFIG['hidden']}")
    print(f"  seq_len: {CONFIG['seq_len']}, horizon: {CONFIG['horizon']}")

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=7, verbose=True
    )
    scaler = GradScaler()

    # Training loop
    print(f"\nEntrenando ({CONFIG['epochs']} epochs max, patience={CONFIG['patience']})...")
    print(f"{'Epoch':>5s} {'Train':>10s} {'Val MAE°C':>10s} {'Val RMSE°C':>11s} "
          f"{'Val R²':>8s} {'LR':>10s} {'Tiempo':>8s}")
    print("-" * 70)

    best_val_loss = float("inf")
    epochs_no_improve = 0
    history = []

    for epoch in range(1, CONFIG["epochs"] + 1):
        t0 = time.time()

        train_loss = train_epoch(
            model, loaders["train"], optimizer, scaler,
            edge_index, edge_weight, CONFIG["n_nodes"]
        )

        val_metrics = evaluate(
            model, loaders["val"], edge_index, edge_weight,
            CONFIG["n_nodes"], scalers, CONFIG["target_idx"]
        )

        elapsed = time.time() - t0
        lr = optimizer.param_groups[0]["lr"]

        print(f"{epoch:5d} {train_loss:10.4f} {val_metrics['MAE_C']:10.4f} "
              f"{val_metrics['RMSE_C']:11.4f} {val_metrics['R2']:8.4f} "
              f"{lr:10.6f} {elapsed:7.1f}s")

        history.append({
            "epoch": epoch, "train_loss": train_loss,
            **{f"val_{k}": v for k, v in val_metrics.items()},
            "lr": lr
        })

        scheduler.step(val_metrics["loss_norm"])

        if val_metrics["loss_norm"] < best_val_loss:
            best_val_loss = val_metrics["loss_norm"]
            epochs_no_improve = 0
            torch.save(model.state_dict(), "best_model.pt")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= CONFIG["patience"]:
                print(f"\nEarly stopping en epoch {epoch}")
                break

    # Evaluación final en test
    print("\n" + "=" * 60)
    print("EVALUACIÓN FINAL EN TEST")
    print("=" * 60)

    model.load_state_dict(torch.load("best_model.pt"))
    test_metrics = evaluate(
        model, loaders["test"], edge_index, edge_weight,
        CONFIG["n_nodes"], scalers, CONFIG["target_idx"]
    )

    print(f"  MAE:  {test_metrics['MAE_C']:.4f} °C")
    print(f"  RMSE: {test_metrics['RMSE_C']:.4f} °C")
    print(f"  R²:   {test_metrics['R2']:.4f}")

    # Guardar historial
    df_hist = pd.DataFrame(history)
    df_hist.to_csv("training_history.csv", index=False)
    print(f"\nHistorial guardado: training_history.csv")
    print(f"Mejor modelo guardado: best_model.pt")

    if torch.cuda.is_available():
        mem_alloc = torch.cuda.max_memory_allocated() / 1e9
        mem_reserved = torch.cuda.max_memory_reserved() / 1e9
        print(f"\nMemoria GPU pico: {mem_alloc:.2f} GB allocated, {mem_reserved:.2f} GB reserved")

ENTRENAMIENTO A3T-GCN - REMMAQ (OPTIMIZADO)
Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
VRAM: 6.4 GB

Cargando artefactos...
  train: 122,712 ventanas
  val  : 17,496 ventanas
  test : 19,657 ventanas

Modelo: A3T-GCN (batched)
  Parámetros entrenables: 27,737
  Hidden: 64
  seq_len: 24, horizon: 1

Entrenando (100 epochs max, patience=15)...
Epoch      Train  Val MAE°C  Val RMSE°C   Val R²         LR   Tiempo
----------------------------------------------------------------------
    1     0.2670     0.8050      1.0887   0.9010   0.001000   461.8s
    2     0.1788     0.6082      0.8349   0.9418   0.001000   463.9s
    3     0.1629     0.5872      0.8153   0.9445   0.001000   464.7s
    4     0.1575     0.5857      0.8190   0.9440   0.001000   464.6s
    5     0.1541     0.5697      0.8032   0.9461   0.001000   466.9s
    6     0.1522     0.5616      0.7845   0.9486   0.001000   464.7s
    7     0.1509     0.5473      0.7723   0.9502   0.001000   466.2s
    8     0.1500     0.

### Arquitectura DCRNN

In [30]:
"""
Entrenamiento DCRNN - REMMAQ (versión optimizada)
Diffusion Convolutional Recurrent Neural Network (Li et al., 2018, ICLR)
Batching a nivel de grafo, misma estructura que train_a3tgcn_fast.py
"""

import numpy as np
import pandas as pd
import pickle
import time
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from torch_geometric_temporal.nn.recurrent import DCRNN

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CONFIG = {
    "n_nodes": 6,
    "in_channels": 14,
    "hidden": 64,
    "seq_len": 24,
    "horizon": 1,
    "target_idx": 6,
    "K": 2,              # orden de difusión (hops en el grafo)
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "epochs": 100,
    "batch_size": 32,
    "patience": 15,
    "seed": 42,
}


def set_seed(seed):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# DATASET (idéntico al de A3T-GCN)
# ============================================================
class STGNNDataset(Dataset):
    def __init__(self, data, mask, seq_len, horizon, target_idx):
        self.data = data
        self.mask = mask
        self.seq_len = seq_len
        self.horizon = horizon
        self.target_idx = target_idx
        self.n_samples = data.shape[0] - seq_len - horizon + 1

    def __len__(self):
        return self.n_samples

    def __getitem__(self, i):
        X = self.data[i:i + self.seq_len]
        y = self.data[i + self.seq_len:i + self.seq_len + self.horizon,
                      :, self.target_idx]
        m = self.mask[i + self.seq_len:i + self.seq_len + self.horizon,
                      :, self.target_idx]
        return (torch.tensor(X, dtype=torch.float32),
                torch.tensor(y, dtype=torch.float32),
                torch.tensor(m, dtype=torch.float32))


# ============================================================
# BATCHING DE GRAFO (idéntico al de A3T-GCN)
# ============================================================
def batch_edge_index(edge_index, batch_size, n_nodes):
    edges = []
    for i in range(batch_size):
        edges.append(edge_index + i * n_nodes)
    return torch.cat(edges, dim=1)


def batch_edge_weight(edge_weight, batch_size):
    return edge_weight.repeat(batch_size)


# ============================================================
# MODELO DCRNN
# ============================================================
class DCRNNForecaster(nn.Module):
    """DCRNN: Difusión (random walk bidireccional) + GRU.
    A diferencia de A3T-GCN que recibe toda la secuencia de golpe,
    DCRNN procesa un timestep a la vez y mantiene estado oculto."""

    def __init__(self, in_channels, hidden, horizon, K=2):
        super().__init__()
        self.dcrnn = DCRNN(in_channels=in_channels, out_channels=hidden, K=K)
        self.head = nn.Linear(hidden, horizon)

    def forward(self, x_seq, edge_index, edge_weight):
        # x_seq: [seq_len, total_nodes, in_channels]
        h = None
        for t in range(x_seq.size(0)):
            h = self.dcrnn(x_seq[t], edge_index, edge_weight, H=h)
        # h: [total_nodes, hidden] (estado oculto del último paso)
        return self.head(h)  # [total_nodes, horizon]


# ============================================================
# LOSS ENMASCARADO
# ============================================================
def masked_mae_loss(pred, target, mask):
    diff = torch.abs(pred - target) * mask
    n_valid = mask.sum()
    if n_valid == 0:
        return torch.tensor(0.0, device=pred.device, requires_grad=True)
    return diff.sum() / n_valid


# ============================================================
# ENTRENAMIENTO
# ============================================================
def train_epoch(model, loader, optimizer, scaler, edge_index_base,
                edge_weight_base, n_nodes):
    model.train()
    total_loss = 0.0
    n_batches = 0
    optimizer.zero_grad()

    for X, y, m in loader:
        bs = X.size(0)
        X = X.to(DEVICE)
        y = y.to(DEVICE)
        m = m.to(DEVICE)

        ei = batch_edge_index(edge_index_base, bs, n_nodes).to(DEVICE)
        ew = batch_edge_weight(edge_weight_base, bs).to(DEVICE)

        with autocast():
            # X: [batch, seq_len, nodes, feat]
            # Reorganizar a [seq_len, batch*nodes, feat] para DCRNN
            x_in = X.permute(1, 0, 2, 3)                  # [seq_len, batch, nodes, feat]
            x_in = x_in.reshape(X.size(1), bs * n_nodes, -1)  # [seq_len, batch*nodes, feat]

            out = model(x_in, ei, ew)                       # [batch*nodes, horizon]
            out = out.reshape(bs, n_nodes, -1)              # [batch, nodes, horizon]
            out = out.permute(0, 2, 1)                      # [batch, horizon, nodes]

            loss = masked_mae_loss(out, y, m)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / n_batches


# ============================================================
# EVALUACIÓN
# ============================================================
@torch.no_grad()
def evaluate(model, loader, edge_index_base, edge_weight_base, n_nodes,
             scalers, target_idx):
    model.eval()
    all_pred, all_target, all_mask = [], [], []

    for X, y, m in loader:
        bs = X.size(0)
        X = X.to(DEVICE)

        ei = batch_edge_index(edge_index_base, bs, n_nodes).to(DEVICE)
        ew = batch_edge_weight(edge_weight_base, bs).to(DEVICE)

        with autocast():
            x_in = X.permute(1, 0, 2, 3)
            x_in = x_in.reshape(X.size(1), bs * n_nodes, -1)
            out = model(x_in, ei, ew)
            out = out.reshape(bs, n_nodes, -1).permute(0, 2, 1)

        all_pred.append(out.cpu())
        all_target.append(y)
        all_mask.append(m)

    pred = torch.cat(all_pred)
    target = torch.cat(all_target)
    mask = torch.cat(all_mask)

    loss_norm = masked_mae_loss(pred, target, mask).item()

    sc = scalers[target_idx]
    pred_np = pred.numpy().flatten()
    target_np = target.numpy().flatten()
    mask_np = mask.numpy().flatten().astype(bool)

    pred_real = sc.inverse_transform(pred_np.reshape(-1, 1)).flatten()
    target_real = sc.inverse_transform(target_np.reshape(-1, 1)).flatten()
    pred_real = pred_real[mask_np]
    target_real = target_real[mask_np]

    mae = np.mean(np.abs(target_real - pred_real))
    rmse = np.sqrt(np.mean((target_real - pred_real) ** 2))
    ss_res = np.sum((target_real - pred_real) ** 2)
    ss_tot = np.sum((target_real - target_real.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot

    return {"loss_norm": loss_norm, "MAE_C": mae, "RMSE_C": rmse, "R2": r2}


# ============================================================
# EJECUCIÓN
# ============================================================
if __name__ == "__main__":
    set_seed(CONFIG["seed"])

    print("=" * 60)
    print("ENTRENAMIENTO DCRNN - REMMAQ (OPTIMIZADO)")
    print("=" * 60)
    print(f"Device: {DEVICE}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

    print("\nCargando artefactos...")
    edge_index = torch.load("edge_index.pt")
    edge_weight = torch.load("edge_weight.pt")

    with open("scalers.pkl", "rb") as f:
        scalers = pickle.load(f)

    loaders = {}
    for split in ["train", "val", "test"]:
        data = np.load(f"tensor_{split}.npy")
        mask = np.load(f"mask_{split}.npy")
        ds = STGNNDataset(data, mask, CONFIG["seq_len"], CONFIG["horizon"],
                          CONFIG["target_idx"])
        shuffle = (split == "train")
        loaders[split] = DataLoader(
            ds, batch_size=CONFIG["batch_size"], shuffle=shuffle,
            num_workers=2, pin_memory=True, drop_last=False
        )
        print(f"  {split:5s}: {len(ds):,} ventanas")

    # Modelo
    model = DCRNNForecaster(
        in_channels=CONFIG["in_channels"],
        hidden=CONFIG["hidden"],
        horizon=CONFIG["horizon"],
        K=CONFIG["K"]
    ).to(DEVICE)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nModelo: DCRNN (K={CONFIG['K']})")
    print(f"  Parámetros entrenables: {n_params:,}")
    print(f"  Hidden: {CONFIG['hidden']}")
    print(f"  seq_len: {CONFIG['seq_len']}, horizon: {CONFIG['horizon']}")

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=7, verbose=True
    )
    scaler = GradScaler()

    print(f"\nEntrenando ({CONFIG['epochs']} epochs max, patience={CONFIG['patience']})...")
    print(f"{'Epoch':>5s} {'Train':>10s} {'Val MAE°C':>10s} {'Val RMSE°C':>11s} "
          f"{'Val R²':>8s} {'LR':>10s} {'Tiempo':>8s}")
    print("-" * 70)

    best_val_loss = float("inf")
    epochs_no_improve = 0
    history = []

    for epoch in range(1, CONFIG["epochs"] + 1):
        t0 = time.time()

        train_loss = train_epoch(
            model, loaders["train"], optimizer, scaler,
            edge_index, edge_weight, CONFIG["n_nodes"]
        )

        val_metrics = evaluate(
            model, loaders["val"], edge_index, edge_weight,
            CONFIG["n_nodes"], scalers, CONFIG["target_idx"]
        )

        elapsed = time.time() - t0
        lr = optimizer.param_groups[0]["lr"]

        print(f"{epoch:5d} {train_loss:10.4f} {val_metrics['MAE_C']:10.4f} "
              f"{val_metrics['RMSE_C']:11.4f} {val_metrics['R2']:8.4f} "
              f"{lr:10.6f} {elapsed:7.1f}s")

        history.append({
            "epoch": epoch, "train_loss": train_loss,
            **{f"val_{k}": v for k, v in val_metrics.items()},
            "lr": lr
        })

        scheduler.step(val_metrics["loss_norm"])

        if val_metrics["loss_norm"] < best_val_loss:
            best_val_loss = val_metrics["loss_norm"]
            epochs_no_improve = 0
            torch.save(model.state_dict(), "best_model_dcrnn.pt")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= CONFIG["patience"]:
                print(f"\nEarly stopping en epoch {epoch}")
                break

    # Evaluación final
    print("\n" + "=" * 60)
    print("EVALUACIÓN FINAL EN TEST")
    print("=" * 60)

    model.load_state_dict(torch.load("best_model_dcrnn.pt"))
    test_metrics = evaluate(
        model, loaders["test"], edge_index, edge_weight,
        CONFIG["n_nodes"], scalers, CONFIG["target_idx"]
    )

    print(f"  MAE:  {test_metrics['MAE_C']:.4f} °C")
    print(f"  RMSE: {test_metrics['RMSE_C']:.4f} °C")
    print(f"  R²:   {test_metrics['R2']:.4f}")

    # Comparación directa con A3T-GCN
    print(f"\nComparación (horizonte 1h):")
    print(f"  {'Modelo':15s} {'MAE°C':>8s} {'RMSE°C':>8s} {'R²':>8s}")
    print(f"  {'A3T-GCN':15s} {'0.5251':>8s} {'0.7522':>8s} {'0.9544':>8s}")
    print(f"  {'DCRNN':15s} {test_metrics['MAE_C']:8.4f} {test_metrics['RMSE_C']:8.4f} {test_metrics['R2']:8.4f}")

    df_hist = pd.DataFrame(history)
    df_hist.to_csv("training_history_dcrnn.csv", index=False)
    print(f"\nHistorial guardado: training_history_dcrnn.csv")
    print(f"Mejor modelo guardado: best_model_dcrnn.pt")

    if torch.cuda.is_available():
        mem = torch.cuda.max_memory_allocated() / 1e9
        print(f"\nMemoria GPU pico: {mem:.2f} GB")

ENTRENAMIENTO DCRNN - REMMAQ (OPTIMIZADO)
Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
VRAM: 6.4 GB

Cargando artefactos...
  train: 122,712 ventanas
  val  : 17,496 ventanas
  test : 19,657 ventanas

Modelo: DCRNN (K=2)
  Parámetros entrenables: 60,161
  Hidden: 64
  seq_len: 24, horizon: 1

Entrenando (100 epochs max, patience=15)...
Epoch      Train  Val MAE°C  Val RMSE°C   Val R²         LR   Tiempo
----------------------------------------------------------------------
    1     0.1236     0.4125      0.6043   0.9695   0.001000   785.4s
    2     0.1100     0.4082      0.5880   0.9711   0.001000   798.3s
    3     0.1067     0.4066      0.5946   0.9705   0.001000   827.7s
    4     0.1053     0.4063      0.5990   0.9700   0.001000   816.7s
    5     0.1040     0.3923      0.5784   0.9720   0.001000   841.0s
    6     0.1029     0.4106      0.6034   0.9696   0.001000   730.1s
    7     0.1226     0.4773      0.6921   0.9600   0.001000   700.6s
    8     0.1193     0.4273    